# 0 Initialization

In [368]:
from pprint import pprint
import re
def extract_python_block(code):
    py_pattern = r"```python\s+(.*?)```"
    clean_code = re.findall(py_pattern, code, re.DOTALL)[-1]
    return clean_code

## 0-1 Loading Data

## 0-2 Model Initialization

In [169]:
from openai import OpenAI
import requests

class LLM:
    def __init__(self, base_url="http://localhost:4869/v1", api_key="EMPTY"):
        self.base_url = base_url
        self.api_key = api_key
        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
        )
        self.health_check()

    def generate(self, data, model='qwen2.5-3b', args=None):
        chat_response = self.client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": data},
            ],
            max_tokens=args['max_tokens'],
            temperature=args['temperature'],
            top_p=args['top_p'],
        )
        return chat_response.choices[0].message.content

    def constrain_generate(self, data, model='qwen2.5-3b', format=None, args=None):
        chat_response = self.client.beta.chat.completions.parse(
            model=model,
            messages=[
                {"role": "user", "content": data},
            ],
            response_format=format,
            max_tokens=args['max_tokens'],
            temperature=args['temperature'],
            top_p=args['top_p'],
        )
        return chat_response.choices[0].message.parsed
    
    def health_check(self):
        health_url = self.base_url.replace("/v1", "") + "/health"
        response = requests.get(health_url)
        if response.status_code == 200:
            print("LLM API is available.")
        else:
            print(f"LLM API health check failed with status code: {response.status_code}")

In [170]:
llm = LLM()

LLM API is available.


In [98]:
args = {
    "max_tokens": 4096,
    "temperature": 0.1,
    "top_p": 0.8,
}

# 1 Code Correction

## 1-1 Normal Python Code Correction

In [48]:
code_correction_template = """
You are an experienced Python programmer, skilled at fixing incorrect Python code.  
Given a Python program that fails to execute properly and the corresponding error message:  
1. Based on the error type and line number in the error message, identify the location of the code causing the execution failure, then analyze the cause of the error.  
2. Analyze the requirements and objectives of the problematic line or code block, combine this with the cause of the error to correct the given code, and place the fixed, grammatically correct, and executable Python code within a ```python ``` block.

Below are some examples:  
---
## Python Code
print("Hello" + 5)  

## Traceback Information
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
TypeError: can only concatenate str (not "int") to str

## Correction
Analysis:  
According to the error message, the error occurred on Line 1 and was caused by directly adding a str-type string and an int-type number. In this case, we can convert the integer 5 to a string so that concatenation is possible. Here is the corrected code:  
```python
# Correction
print("Hello" + str(5))  
```
---
NOTE:  
- There may be various types of errors in Python programming; please make appropriate corrections based on each error type.  
- Try not to use try-except blocks to catch errors and bypass problematic code, as that is not the intended solution.  
- Strictly follow the format: the corrected Python code must be enclosed within a ```python ``` block.

Based on the above instructions and description, please help fix the following Python program:  
## Python Code
${code}

## Traceback Information
${error}

## Correction
"""

In [53]:
import subprocess
import sys
import re

def run_code(code_string):
    try:
        # 使用当前 python 解释器 (sys.executable) 执行代码
        # capture_output=True 会同时捕获 stdout 和 stderr
        # text=True 会让结果以字符串形式返回，而不是字节
        result = subprocess.run(
            [sys.executable, "-c", code_string],
            capture_output=True,
            text=True,
            timeout=10 # 设置超时防止死循环
        )
        
        if result.returncode == 0:
            return {"success": True, "output": result.stdout, "error": None}
        else:
            # 如果 returncode != 0，说明出错了
            # stderr 通常包含 Traceback
            return {"success": False, "output": result.stdout, "error": result.stderr}
    except subprocess.TimeoutExpired:
        return {"success": False, "error": "RunTimeError: 代码执行超时！"}
    except Exception as e:
        return {"success": False, "error": str(e)}

def refine_code(code, error):
    fix_bug_prompt = code_correction_template
    from string import Template
    fix_T = Template(fix_bug_prompt)
    fix_prompt = fix_T.safe_substitute(code=code, error=error)
    fix_out = llm.generate(data=fix_prompt, args=args)
    # print(fix_out)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, fix_out, re.DOTALL)
    return matches[-1]

def refine_loop(code):
    res = run_code(code)
    while not res["success"]:
        code = refine_code(code,res["error"])
        print("## New code ##")
        print(code)
        res = run_code(code)
        print(res)

# 测试代码
dangerous_code = """
import sys
def foo():
    bar(1,0)
def bar(x,y):
    a = x/y
foo()
"""

res = run_code(dangerous_code)
if not res["success"]:
    print(res["error"])

refine_loop(dangerous_code)


Traceback (most recent call last):
  File "<string>", line 7, in <module>
  File "<string>", line 4, in foo
  File "<string>", line 6, in bar
ZeroDivisionError: division by zero

## New code ##
def foo():
    bar(1, 0)

def bar(x, y):
    if y != 0:
        a = x / y
    else:
        print("Error: Division by zero")
        return None

foo()

{'success': True, 'output': 'Error: Division by zero\n', 'error': None}


## 1-2 Z3 Python Code Correction

In [54]:
z3_code_correction_template = """
You are an experienced Python programmer skilled at fixing erroneous Python code.  
Given a Python program that fails to execute correctly and the corresponding error message, the goal of this Python code is to use Z3 to verify the satisfiability of a translated first-order predicate logic.  

## Coding Principles and Components of the Z3 Python API:  
1. Import and Initialization  
```code
from z3 import *

# Create a solver instance
solver = Solver()
```

2. Declare Variables and Types  
```code
# Boolean variables
p = Bool('p')
q = Bool('q')

# Integer variables
x = Int('x')
y = Int('y')

# Real variables
r = Real('r')

# Bit vectors
bv = BitVec('bv', 32)  # 32-bit bit vector

# Custom sort (type) - for first-order logic
Sort_Person = DeclareSort('Person')
alice = Const('alice', Sort_Person)
bob = Const('bob', Sort_Person)
```

3. Core Elements of First-Order Predicate Logic  
```code
# Declare functions (predicates)
# Unary predicate
IsHappy = Function('IsHappy', Sort_Person, BoolSort())

# Binary predicate (relation)
Loves = Function('Loves', Sort_Person, Sort_Person, BoolSort())

# Function symbols
Father = Function('Father', Sort_Person, Sort_Person)

# Quantifier variables
x_var = Const('x', Sort_Person)
y_var = Const('y', Sort_Person)
```

4. Construct Formulas  
```code
# Propositional logic connectives
f1 = And(p, q)           # Conjunction
f2 = Or(p, q)            # Disjunction
f3 = Not(p)              # Negation
f4 = Implies(p, q)       # Implication
f5 = p == q              # Equivalence

# First-order logic quantifiers
# Universal quantifier: ∀x. IsHappy(x)
f6 = ForAll([x_var], IsHappy(x_var))

# Existential quantifier: ∃x. IsHappy(x)
f7 = Exists([x_var], IsHappy(x_var))

# Multiple quantifier variables
# ∀x. ∃y. Loves(x, y)
f8 = ForAll([x_var], Exists([y_var], Loves(x_var, y_var)))

# Complex formula
# ∀x. (IsHappy(x) → ∃y. Loves(x, y))
f9 = ForAll([x_var], 
    Implies(IsHappy(x_var), 
            Exists([y_var], Loves(x_var, y_var))))
```

5. Constraints and Solving  
```code
# Add constraints to the solver
solver.add(f6)
solver.add(f9)

# Check satisfiability
result = solver.check()

if result == sat:
    print("Satisfiable!")
    model = solver.model()
    print(model)
elif result == unsat:
    print("unsat")
else:
    print("unknow")
```

## Task Goal  
1. Based on the error type and line number in the error message, locate the code position causing the execution failure, then analyze the cause of the error.  
2. Analyze the requirements and objectives of the problematic line or code block, and use the coding principles to identify the root cause of the error. Correct the given code accordingly, ensuring the resulting code is syntactically correct and executable, then place the corrected Python code within a ```python ``` block.

## NOTE  
- In Python programming, there may be various types of errors; please address each type appropriately.  
- Avoid using try-except blocks to catch errors as a workaround; this is not the intended solution.  
- Strictly follow the format requirement: the corrected Python code must be placed within a ```python ``` block.

Based on the above guidelines and task description, please help me fix the following Python program:  
## Python Code  
${code}  

## Traceback Information  
${error}  

## Correction
"""

In [392]:
def wrap_z3_code(declaration, expression):
    z3_code = "# Auto-generated Z3 code\n\n"
    z3_code += "from z3 import *\n\n"
    z3_code += "s = Solver()\n\n"
    z3_code += "s.reset()"
    z3_code += "# --- Declarations ---\n\n"
    z3_code += declaration + "\n\n"
    z3_code += "# --- Expressions ---\n\n"
    z3_code += expression + "\n\n"
    z3_code += "result = s.check()\n"
    z3_code += "print(f'Result: {result}')\n"
    z3_code += "if result == sat:\n"
    z3_code += "    print('Model:', s.model())\n"
    return z3_code

def correct_z3_code(code, error):
    fix_bug_prompt = z3_code_correction_template
    from string import Template
    fix_T = Template(fix_bug_prompt)
    fix_prompt = fix_T.safe_substitute(code=code, error=error)
    fix_out = llm.generate(data=fix_prompt, args=args)
    print(fix_out)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, fix_out, re.DOTALL)
    return matches[-1]

def correct_loop(code):
    res = run_code(code)
    print(res)
    while not res["success"]:
        code = correct_z3_code(code,res["error"])
        res = run_code(code)
        print(res)

In [58]:
declaration = ""
expression = ""
code = wrap_z3_code(declaration,expression)
correct_loop(code)

{'success': True, 'output': 'Result: sat\nModel: []\n', 'error': None}


# Pilot Experiment

In [81]:
declaration_gen_template = """
You are an expert in formal logic and automated reasoning. Your task is to extract a non-redundant logical schema from a research hypothesis and translate it into a rich set of Z3 Python declarations.

Instructions:

   1.  Entity Extraction & Uniqueness:
      •  Use EnumSort for any set of entities that are mutually exclusive and represent a fixed range of choices (e.g., specific Universities, Ethnicities, or Statuses).
      •  Use DeclareSort ONLY for open-ended domains where the number of individuals is not fixed (e.g., Person).
      •  Rule: If a group of entities is defined via EnumSort, DO NOT declare a separate Sort or separate Const objects for those entities.
      •  Constants: Only declare Const for specific individuals (e.g., zhao_ming) or unique regions that are not part of a fixed category.


   2.  Diverse Function Declarations:
      •  Predicates: Functions returning `BoolSort()` for properties (e.g., `IsWesternized`).
      •  Numerical Functions: Functions returning `RealSort()` or `IntSort()` for measurable attributes (e.g., `BloodPressure(p)`, `SaltConcentration(r)`, `Age(p)`).
      •  Mapping Functions: Functions returning a custom Sort (e.g., `Origin(p) -> Region`, `CurrentDiet(p) -> DietType`).

   3.  Strict Syntax Rules:
      •  Use `DeclareSort`, `EnumSort`, `Function`, `Const`, `RealSort`, `IntSort`, and `BoolSort`.
      •  Declare universally quantified variables (e.g., `x`, `y`, `r`) for all defined sorts.
      •  CRITICAL: Each logical concept must have exactly ONE representation. Do not define the same property as both a Predicate and an Enum/Function.
      •  CRITICAL: Do NOT include any `solver.add(...)` or logical axioms. The output must consist ONLY of the structural declarations and signatures.

   4.  Output Format:
      •  Output ONLY a Python code block. No natural language or commentary.
      •  Organize by category: Sorts/Enums, Variables, Constants, and Functions.

Example Input:

<Context>
Black Americans are twice as likely to suffer from hypertension as white Americans. The same is true when comparing Westernized black Africans to white Africans. The researchers hypothesized that the reason why westernized black people suffer from hypertension is the result of the interaction of two reasons: one is the high salt content of western foods, and the other is the adaptation mechanism of black genetic genes to the salt-deficient environment.</Context>
<Question>
The following conclusions about contemporary westernized African blacks, if the item is true, can it best support the researchers' hypothesis?
</Question>
<Options>
Option (A): The blood pressure of the descendants of Senegalese and Gambians is usually not high, and the history of Senegal and Gambia has not been short of salt.
Option (B): The unusually high salt intake in certain parts of Africa is a serious problem that threatens the health of residents.
Option (C): Considering health care, most African whites also pay attention to controlling salt intake.
Option (D): The blood pressure of Yoruba people in West Africa is not high. Yoruba people have lived inland far away from sea salt and far away from the Sahara salt mine in Africa.
</Options>

Example Output:

```python
from z3 import *

# 1. Declare Sorts & Enums
Person = DeclareSort('Person')
Region = DeclareSort('Region')
Ethnicity, (Black, White, Other) = EnumSort('Ethnicity', ['Black', 'White', 'Other']) # Not Ethnicity = EnumSort('Ethnicity', ['Black', 'White', 'Other'])
DietCategory, (Western, Traditional, Mixed) = EnumSort('DietCategory', ['Western', 'Traditional', 'Mixed'])

# 2. Declare Variables
p = Const('p', Person)
r = Const('r', Region)

# 3. Declare Constants
senegal = Const('senegal', Region)
yoruba_people = Const('yoruba_people', Person)

# 4. Declare Functions with varied Return Sorts
# 4.1 Boolean
IsWesternized = Function('IsWesternized', Person, BoolSort()) 
HasGeneticAdaptation = Function('HasGeneticAdaptation', Person, BoolSort())

# 4.2 Arithmetic
BloodPressureValue = Function('BloodPressureValue', Person, RealSort())
SaltIntakeLevel = Function('SaltIntakeLevel', Person, RealSort())
HistoricalSaltAbundance = Function('HistoricalSaltAbundance', Region, RealSort())

# 4.3 Object Mapping
OriginRegion = Function('OriginRegion', Person, Region)
AssignedDiet = Function('AssignedDiet', Person, DietCategory)
GetEthnicity = Function('GetEthnicity', Person, Ethnicity)
```

Task: given the provided context and options, generate the corresponding rich Z3 declarations.
"""

In [137]:
decl_content = """
<Context>
${context}
</Context>
<Question>
${question}
</Question>
<Options>
${options}
</Options>
"""
from string import Template
declT = Template(decl_content)
decl_content = declT.safe_substitute(context=context, question=question, options= options)
decl_content

'\n<Context>\nIn the planning of a new district in a township, it was decided to build a special community in the southeast, northwest, centered on the citizen park. These four communities are designated as cultural area, leisure area, commercial area and administrative service area. It is known that the administrative service area is southwest of the cultural area, and the cultural area is southeast of the leisure area.\n</Context>\n<Question>\nBased on the above statement, which of the following can be derived?\n</Question>\n<Options>\n(A) Civic Park is north of the administrative service area.\n(B) The leisure area is southwest of the cultural area.\n(C) The cultural district is in the northeast of the business district.\n(D) The business district is southeast of the leisure area.\n</Options>\n'

In [138]:
decl_res = llm.generate(data=declaration_gen_template + decl_content, args=args)

In [139]:
py_pattern = r"```python\s+(.*?)```"
decl_code = re.findall(py_pattern, decl_res, re.DOTALL)[-1]

In [140]:
pprint(decl_code)

('from z3 import *\n'
 '\n'
 '# 1. Declare Sorts & Enums\n'
 "Area = DeclareSort('Area')\n"
 'Direction, (Northwest, Southeast, Southwest, Northeast) = '
 "EnumSort('Direction', ['Northwest', 'Southeast', 'Southwest', 'Northeast'])\n"
 '\n'
 '# 2. Declare Variables\n'
 "cultural_area = Const('cultural_area', Area)\n"
 "leisure_area = Const('leisure_area', Area)\n"
 "commercial_area = Const('commercial_area', Area)\n"
 "administrative_service_area = Const('administrative_service_area', Area)\n"
 '\n'
 '# 3. Declare Constants\n'
 "citizen_park = Const('citizen_park', Area)\n"
 '\n'
 '# 4. Declare Functions with varied Return Sorts\n'
 '# 4.1 Boolean\n'
 "IsSouthwestOf = Function('IsSouthwestOf', Area, Area, BoolSort())\n"
 "IsSoutheastOf = Function('IsSoutheastOf', Area, Area, BoolSort())\n"
 '\n'
 '# 4.2 Object Mapping\n'
 "LocationRelativeToPark = Function('LocationRelativeToPark', Area, Direction, "
 'BoolSort())\n')


In [142]:
res = run_code(decl_code)

In [143]:
res

{'success': True, 'output': '', 'error': None}

# Declaration Generation

In [144]:
Implication_template = """

Role: 

    You are a Formal Logic Translator specializing in converting natural language reasoning into executable Z3 Python code. Your goal is to take a structured logical argument and map it onto a provided Z3 declaration schema.

Input Data:

    1.  Context & Question: The background information and the target problem.

    2.  Z3 Declarations: The predefined Sorts, Functions, and Predicates.

    3.  Reasoning Steps: A sequence of <step> blocks containing <premise> and <conclusion> in natural language.

Translation Rules (Strict Compliance Required):

    1.  Strict Symbolic Mapping:

        • Use ONLY the functions, predicates, and sorts provided in the Z3 Declaration.

        • Use ForAll, Exists, And, Or, Not, Implies, and == for logical connectors.

    2.  Explicit Expression Rule (No Pointers):

        • DO NOT refer to previous steps using variable names like conclusion_n or step_n.

        • Instead, you must explicitly restate the full Z3 logical expression of that conclusion as a premise in the new step.

    3.  Grounded Premises:

        • Every symbolic statement in premises_n must be derived from:

            ◦ The current step's natural language <premise>.

            ◦ A previous step's <conclusion> (written in full symbolic form).

            ◦ Global background knowledge or definitions implied by the Context.

    4.  Logical Purity:

        • Avoid using True or False as placeholders. Every statement should be a well-formed formula (WFF) using the provided predicates.

        • Ensure the conclusion_n is a direct logical consequence of the premises_n.

    5.  Output Format:

        • Output ONLY the Python code.

        • Structure the output as # Step N, followed by premises_n = [...] and conclusion_n = ....

        • Each step must be separated by exactly one newline.

Example of Expected Behavior:

Question:
<Context>Black Americans are twice as likely to suffer from hypertension as white Americans. The same is true when comparing Westernized black Africans to white Africans. The researchers hypothesized that the reason why westernized black people suffer from hypertension is the result of the interaction of two reasons: one is the high salt content of western foods, and the other is the adaptation mechanism of black genetic genes to the salt-deficient environment.</Context>
<Question>The following conclusions about contemporary westernized African blacks, if the item is true, can it best support the researchers' hypothesis?</Question>
<Options>
Option (A): The blood pressure of the descendants of Senegalese and Gambians is usually not high, and the history of Senegal and Gambia has not been short of salt.
Option (B): The unusually high salt intake in certain parts of Africa is a serious problem that threatens the health of residents.
Option (C): Considering health care, most African whites also pay attention to controlling salt intake.
Option (D): The blood pressure of Yoruba people in West Africa is not high. Yoruba people have lived inland far away from sea salt and far away from the Sahara salt mine in Africa.
</Options>

Z3 Declaration:
from z3 import *

# Declare sorts
Person = DeclareSort('Person')
Region = DeclareSort('Region')

# Declare predicates
Black = Function('Black', Person, BoolSort())
Westernized = Function('Westernized', Person, BoolSort())
FromRegion = Function('FromRegion', Person, Region, BoolSort())
SaltSufficientHistory = Function('SaltSufficientHistory', Region, BoolSort())
SaltDeficientHistory = Function('SaltDeficientHistory', Region, BoolSort())
GeneticAdaptationToLowSalt = Function('GeneticAdaptationToLowSalt', Person, BoolSort())
HighSaltDiet = Function('HighSaltDiet', Person, BoolSort())
Hypertension = Function('Hypertension', Person, BoolSort())

Reasoning steps:
<step>
<premise>Descendants of Senegalese (a) are Westernized black individuals.</premise>
<conclusion>Individual 'a' is Black and Westernized.</conclusion>
</step>
<step>
<premise>Individual 'a' is Black and Westernized.</premise>
<premise>The hypothesis states all Westernized Black people with high salt diets and genetic adaptation get hypertension.</premise>
<conclusion>Hypertension(a) == And(HighSaltDiet(a), GeneticAdaptation(a))</conclusion>
</step>

Example Output:
# Step 1
a = Const('a', Person)
premises_1 = [
    Black(a),
    Westernized(a)
]
conclusion_1 = And(Black(a), Westernized(a))

# Step 2=
premises_2 = [
    And(Black(a), Westernized(a)),
    ForAll([x], Implies(And(Black(x), Westernized(x)), Hypertension(x) == And(HighSaltDiet(x), GeneticAdaptation(x))))
]
conclusion_2 = (Hypertension(a) == And(HighSaltDiet(a), GeneticAdaptation(a)))
"""

# Generate

In [312]:
gen_template = """
System Role:

    You are an expert Logical Reasoner and Formal Methods Specialist. Your task is to decompose complex research contexts and multiple-choice options into an atomic reasoning chain. This chain must be structured such that each individual step represents a single, valid logical inference suitable for translation into First-Order Logic (FOL) or Z3 SMT-solver syntax.

Operational Rules:

    1. Atomic Inference Rule: Each <step> must contain premises that lead to exactly one specific <conclusion>. If a set of premises implies multiple conclusions, you must break them into multiple, sequential <step> blocks.

    2. Flexible Premise Count: A <step> can have a single premise or multiple premises.

    3. Sequential Continuity: The conclusion of a previous step should serve as a premise for subsequent steps where necessary to maintain the causal link.

    4. Strict Formatting: 
        • Every <step> block must be separated by exactly two newlines (\n\n).

        • The flow should be a continuous logical progression through all relevant premises provided in the context and options.

    5. Content Constraints:
        • Use <premise>...</premise> for facts, definitions, or previous conclusions.

        • Use <conclusion>...</conclusion> for the immediate logical output of that step.

    6. Final Evaluation: After the full chain, provide a brief synthesis of which option is logically consistent with the hypothesis, followed by the answer in \boxed{}.

Output Template Example:

<step>
<premise>High salt environment exists.</premise>
<premise>If environment is high salt, then individuals consume more salt.</premise>
<conclusion>Individuals in Context A consume more salt.</conclusion>
</step>

<step>
<premise>Individuals in Context A consume more salt.</premise>
<premise>Excess salt consumption is a necessary condition for hypertension.</premise>
<conclusion>Individuals in Context A meet one necessary condition for hypertension.</conclusion>
</step>

[Synthesis of all options and logical nodes]

\boxed{Letter}
"""

In [320]:
rephrase_content ="""
<Context>
${context}
</Context>
<Question>
${question}
</Question>
<Options>
${options}
</Options>
"""
from string import Template
rephraseT = Template(rephrase_content)
context = "Zhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct."
question="Well, their admission status is."
options="""
Option (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.

Option (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.

Option (C):Zhao Ming, Qian Hong and Sun Jie were admitted to Beijing Normal University, Tsinghua University and Peking University respectively.

Option (D):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Beijing Normal University and Tsinghua University respectively.
"""
input_rephrase = rephraseT.safe_substitute(context=context, question=question, options= options)
input_rephrase

"\n<Context>\nZhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct.\n</Context>\n<Question>\nWell, their admission status is.\n</Question>\n<Options>\n\nOption (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.\n\nOption (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.\n\nOption (C

In [321]:
gen_res = llm.generate(data=gen_template + input_rephrase, args=args)

In [319]:
pprint(gen_res)

('<step>\n'
 "<premise>Each student's guess had exactly one correct statement.</premise>\n"
 "<conclusion>At least one student's guess has exactly one correct "
 'statement.</conclusion>\n'
 '</step>\n'
 '\n'
 '<step>\n'
 '<premise>Classmate A guessed that Zhao Ming was admitted to Tsinghua '
 'University and Sun Jie was admitted to Beijing Normal University.</premise>\n'
 '<conclusion>Either Zhao Ming was admitted to Tsinghua University or Sun Jie '
 'was admitted to Beijing Normal University, but not both.</conclusion>\n'
 '</step>\n'
 '\n'
 '<step>\n'
 '<premise>Student B guessed that Zhao Ming was admitted to Beijing Normal '
 'University and Qian Hong was admitted to Tsinghua University.</premise>\n'
 '<conclusion>Either Zhao Ming was admitted to Beijing Normal University or '
 'Qian Hong was admitted to Tsinghua University, but not both.</conclusion>\n'
 '</step>\n'
 '\n'
 '<step>\n'
 '<premise>Student C guessed that Zhao Ming was admitted to Peking University '
 'and Sun Jie was

In [324]:
def get_step_list(text_content):
    Steps = []
    pattern = r"<step.*?>(.*?)</step>"
    matches = re.findall(pattern, text_content, flags=re.DOTALL)
    for i, content in enumerate(matches, 1):
        clean_content = content.strip()
        Steps.append(clean_content)
    return Steps
step_list = get_step_list(gen_res)
step_list[0]

"<premise>Classmate A guessed Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.</premise>\n<premise>Classmate B guessed Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.</premise>\n<premise>Classmate C guessed Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.</premise>\n<premise>The students' guesses were half correct.</premise>\n<conclusion>At least one student's guess is partially correct.</conclusion>"

In [326]:
def get_premise_conclusion(step_content):
    premise_list = []
    pattern = r"<premise>(.*?)</premise>"
    matches = re.findall(pattern, step_content, flags=re.DOTALL)
    if matches:
        for i, content in enumerate(matches, 1):
            clean_content = content.strip()
            premise_list.append(clean_content)

    pattern = r"<conclusion>(.*?)</conclusion>"
    matches = re.findall(pattern, step_content, flags=re.DOTALL)
    conclusion = matches[-1] if matches else None
    return premise_list, conclusion
premise_list, conclusion = get_premise_conclusion(step_list[0])
premise_list, conclusion

(['Classmate A guessed Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.',
  'Classmate B guessed Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.',
  'Classmate C guessed Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.',
  "The students' guesses were half correct."],
 "At least one student's guess is partially correct.")

# 1 Rephrase & Knowledge Injection

In [301]:
rephrase_template = """
You are an experienced linguist with strong natural language understanding and reasoning capabilities.

### Task Description
To make logical reasoning problems easier for solvers to understand and more suitable for verifying solution steps using the Python Z3 API, your goal consists of two steps. Please strictly follow these two steps for analysis and rewriting the problem:
Step 1: Rewrite the problem context. Follow these principles:
- Remove questions, as interrogative sentences cannot form propositions in propositional logic;
- Eliminate irrelevant background or introductory information, keeping only key constraints needed for subsequent reasoning;
- IMPORTANT: Avoid using colons, dashes, and other punctuation in rewritten sentences, as they can complicate sentence structure. Prefer simple sentence structures when rewriting key sentences related to constraints;
- IMPORTANT: Ensure the semantic meaning of the rewritten sentences is consistent with the original;

Step 2: Coreference resolution. When referring across sentences, replace pronouns with their specific antecedents to eliminate ambiguity in subsequent sentence splitting.

Step 3: Inject common sense and axioms. We assume solvers lack common sense and axioms. Therefore, based on the context and question description, provide potentially needed common sense and axioms. Examples include:
- Common sense: "The morning sun is in the east," "If A is west of B, then B is east of A";
- Axioms: "If x, y, z are real numbers and a > b, b > c, then a > c," "The sum of angles in a triangle is 180 degrees."

## Output Format
1. IMPORTANT: Output consists of two paragraphs. The first paragraph contains the rewritten context, and the second paragraph contains the common sense and axioms required for solving the problem.
2. Separate the two paragraphs with\n\n;
3. Only rewrite the context;

Here is an example:

## Input
<Context>
The law firm has eight partners: Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens. From 1961 to 1968, one partner joined the firm each year. Hodges joined the firm earlier than Nader. King joined earlier than James. Both Nader and James joined earlier than Gregg. Nader joined earlier than Owens. James joined earlier than MacNeil. Gregg joined earlier than Ivan.
</Context>
<Question>
Which of the following statements cannot be true? </Question>
<Options>
(A) Hodges joined the firm in 1961.
(B) Hodges joined the firm in 1963.
(C) Gregg joined the firm in 1964.
(D) MacNeil joined the firm in 1964.
(E) Owens joined the firm in 1964.
</Options>

## Output
<p1>Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens is the partners of the law firm.</p1>
<p2>Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens joined the firm between 1961 to 1968</p2>
<p3>Each partner joined the firm in different year.</p3>
<p4> Hodges joined earlier than Nader </p4>
<p5> King joined earlier than James. </p5>
<p6> Nader joined earlier than Gregg</p6>
<p7> James joined earlier than Gregg</p7>
<p8> Nader joined earlier than Owens</p8>
<p9> James joined earlier than MacNeil.</p9>
<p10>Gregg joined earlier than Ivan.</p10>

<axiom_1> If $a$ joined earlier than $b$, and $b$ joined earlier than $c$, then $a$ joined earlier than $c$.</axiom_1>
---
Now, Please rephrase the following text and given the axiom related to the problem:
"""

In [302]:
rephrase_content ="""
<Context>
${context}
</Context>
<Question>
${question}
</Question>
<Options>
${options}
</Options>
"""
from string import Template
rephraseT = Template(rephrase_content)
context = "Zhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct."
question="Well, their admission status is."
options="""
Option (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.

Option (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.

Option (C):Zhao Ming, Qian Hong and Sun Jie were admitted to Beijing Normal University, Tsinghua University and Peking University respectively.

Option (D):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Beijing Normal University and Tsinghua University respectively.
"""
input_rephrase = rephraseT.safe_substitute(context=context, question=question, options= options)
input_rephrase

"\n<Context>\nZhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct.\n</Context>\n<Question>\nWell, their admission status is.\n</Question>\n<Options>\n\nOption (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.\n\nOption (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.\n\nOption (C

In [305]:
rephrase_res = llm.generate(data= rephrase_template + input_rephrase, args=args)

In [306]:
rephrase_res

"<p1>Zhao Ming, Qian Hong, and Sun Jie were admitted to Peking University, Tsinghua University, and Beijing Normal University.</p1>\n<p2>Classmate A guessed that Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.</p2>\n<p3>Student B guessed that Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.</p3>\n<p4>Student C guessed that Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.</p4>\n<p5>The students' guesses were half correct.</p5>\n\n<atom_1> Each student's guess had exactly one correct statement.</atom_1>"

# 2 Declaration Genaration

##  2.1 Objectives Extraction

In [270]:
ner_template = """
You are an experienced natural language processing expert, familiar with tasks such as named entity recognition and relation extraction.

## Task Description
Given the context, question description, and candidate options of a multiple-choice question, your task is to extract key entities and objectives from the question and determine their types. Please analyze according to the following steps:
1. Determine which entities and objectives are being focused on; they may be people, things, events, or other categories;
2. After obtaining the list of key targets, cluster the entities and objectives. For example: person={John, Marry}, teacher={XiaoMing,Alice};
3. Expand specific targets based on context semantics. For example, "There are 3 lockers," then lockers={locker_1, locker_2, locker_3}
4. The type of the entities and objectives should not be a verb;
5. Whenever possible, use the descriptions found in the text as the entity types.

## Output Format
1. The expected output type is a dictionary;
2. The value in the output dictionary should be lists of targets, with their keys being the types of these targets.
3. The value of the output dictionary can only be a single-layer list, not other data types such as dictionary or nested lists.

Below is a simple example:

## Example
#### Input
<Context>
A law firm has eight partners, namely Gregg, Hodges, Ivan, James, King, MacNeil, Nader and Owens. From 1961 to 1968, a partner joined the firm each year. Hodges joined the firm before Nader. King joined the firm before James. Nader and James joined the firm before Gregg. Nader joined the firm before Owens. James joined the firm before MacNeil. Gregg joined the firm before Ivan.
</Context>
<Question>
Which of the following cannot be true?
</Question>
<Options>
(A) Hodges joined the law firm in 1961.
(B) Hodges joined the law firm in 1963.
(C) Gregg joined the law firm in 1964.
(D) MacNeil joined the law firm in 1964.
(E) Owens joined the law firm in 1964.
</Options>

#### Output
{
    "partner" : ["Gregg", "Hodges", "Ivan", "James", "King", "MacNeil", "Nader"]
    "year" : [1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968]
}
Now, extract the targets and entities from the following text:

"""

In [271]:

from pydantic import BaseModel
from typing import Dict, Set, Any

class DynamicSets(BaseModel):
    data: Dict[str, List[Any]]

# 使用
result = DynamicSets(data={
    "partners": {"Gregg", "Hodges"},
    "years": {1961, 1962},
    "scores": {3.5, 4.0, 4.5}
})


In [272]:
ner_content = """
<Context>
${context}
</Context>
<Question>
${question}
</Question>
<Options>
${options}
</Options>
"""
from string import Template
nerT = Template(ner_content)
context = "Zhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct."
question="Well, their admission status is."
options="""
Option (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.

Option (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.

Option (C):Zhao Ming, Qian Hong and Sun Jie were admitted to Beijing Normal University, Tsinghua University and Peking University respectively.

Option (D):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Beijing Normal University and Tsinghua University respectively.
"""
input_content = nerT.safe_substitute(context=context, question=question, options= options)
input_content

"\n<Context>\nZhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct.\n</Context>\n<Question>\nWell, their admission status is.\n</Question>\n<Options>\n\nOption (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.\n\nOption (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.\n\nOption (C

In [273]:
# res = llm.generate(data= ner_template + input_content, args=args)
res = llm.constrain_generate(data= ner_template + input_content, format=DynamicSets, args=args)

In [274]:
type(res.data)

dict

In [275]:
entities = res.data
entities

{'person': ['Zhao Ming', 'Qian Hong', 'Sun Jie'],
 'university': ['Peking University',
  'Tsinghua University',
  'Beijing Normal University']}

## 2.2 Properties and Predicates Extraction

In [291]:
re_template = """
You are an experienced natural language processing expert with the capability to extract key information, such as entity relation extraction and predicate extraction.

## Task Description
Given the context, question description, multiple-choice options, and a list of extracted entities (in dictionary format, where the key represents the entity type and the value represents the list of entities of that type), your task is to extract key predicates or relations from the provided information. Additionally, clarify which entity types each predicate or relation acts upon.

- Key Predicate: A key predicate represents a crucial action performed by entities in the given text/question. For example, in the following scenario, the question investigates the event of each partner joining the firm. Thus, there is a key predicate **join**, which acts on the "partners" type entities and returns the time of joining (i.e., entities of type "years"). Therefore, the expected output from this example is:
``` 
{
...(other relations)
"join": ["partners", "years"],
}
```

- Key Relation: Here, the relationships to be extracted are the same as those in NLP relation extraction tasks, including positional relations, hierarchical relations, and role-based relations. The difference between relations and predicates is that relations may not necessarily be verbs. For instance, in the sentence "The library is southwest of the classroom," this indicates a positional relationship. This relation can be named "location." Assuming both the library and classroom are of type "building," and "southwest" is of type "Direction," the expected partial output is:
``` 
{
...(other relations)
"location": ["building", "building", "Direction"],
}
```
Here, the relation takes two entities of type "building" as input and returns their relative position (of type "Direction").

## Output Format
1. The expected output is a dictionary (Dictionary);
2. The values in the output dictionary should be target lists, where the corresponding keys represent the extracted key relations and predicates; the values are lists of entity types indicating the input entity types, and the output entity type (by default, the last entity type in the list is assumed to be the return type, as described in the task).
3. The values in the output dictionary must be single-level lists and cannot contain dictionaries or nested lists or any other data types.
4. For relationships that require truth-value judgment, append "bool" as the last element in the list. For example, in "John is taller than Mary," there is a relation "Taller_than," with inputs being two "person" entities and no explicit output type. However, the output type is True or False, so append "Bool" to the list corresponding to "Taller_than":

```
{
...
"Taller_than": ["person", "person", "BoolSort()"],
}
```
5.CRITICAL: in addition to "BoolSort()", "RealSort()" (representing real numbers) and "IntSort()" (representing integers) are also available. When using these potential types, their initial letters must be capitalized!
6.CRITICAL：The values in the output dictionary must be of type List. The elements within the list must be of the entity types defined in the `entity list`, please ensure strict consistency, 


Below is a simple example:

## Example
#### Input
<Context>
The law firm has eight partners: Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens. From 1961 to 1968, one partner joined the firm each year. Hodges joined the firm earlier than Nader. King joined earlier than James. Both Nader and James joined earlier than Gregg. Nader joined earlier than Owens. James joined earlier than MacNeil. Gregg joined earlier than Ivan.
</Context>
<Question>
Which of the following statements cannot be true? </Question>
<Options>
(A) Hodges joined the firm in 1961.
(B) Hodges joined the firm in 1963.
(C) Gregg joined the firm in 1964.
(D) MacNeil joined the firm in 1964.
(E) Owens joined the firm in 1964.
</Options>

<Entity_List>
{
'partners': {'Gregg', 'Hodges', 'Ivan', 'James', 'King', 'MacNeil', 'Nader', 'Owens'},
 'years': {1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968}
}
</Entity_List>

#### Output
{
"join" : ["partner", "year"], # The element in the list should in ["partners","years","BoolSort()","IntSort()","RealSort()"]
"join_before": ["partner", "partner", "BoolSort()"]
} Now, extract targets and entities from the following text:
"""

In [292]:
re_content = """
<Context>
${context}
</Context>
<Question>
${question}
</Question>
<Options>
${options}
</Options>
<Entity_List>
${entities}
</Entity_List>
"""
from string import Template
reT = Template(re_content)
context = "Zhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct."
question="Well, their admission status is."
options="""
Option (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.

Option (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.

Option (C):Zhao Ming, Qian Hong and Sun Jie were admitted to Beijing Normal University, Tsinghua University and Peking University respectively.

Option (D):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Beijing Normal University and Tsinghua University respectively.
"""
input_re = reT.safe_substitute(context=context, question=question, options= options, entities=entities )
input_re

"\n<Context>\nZhao Ming, Qian Hong and Sun Jie were admitted to Peking University, Tsinghua University and Beijing Normal University. Which schools were they admitted to? The students made the following guesses? Classmate A guessed? Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University. Student B guess? Zhao Ming was admitted to Beijing Normal University, Qian Hong was admitted to Tsinghua University. Student C guess? Zhao Ming was admitted to Peking University, Sun Jie was admitted to Tsinghua University. As a result, the students' guesses were half correct.\n</Context>\n<Question>\nWell, their admission status is.\n</Question>\n<Options>\n\nOption (A):Zhao Ming, Qian Hong and Sun Jie were accepted by Peking University, Tsinghua University and Beijing Normal University respectively.\n\nOption (B):Zhao Ming, Qian Hong and Sun Jie were admitted to Tsinghua University, Beijing Normal University and Peking University respectively.\n\nOption (C

In [293]:
re_res = llm.constrain_generate(data= re_template + input_content, format=DynamicSets, args=args)

In [294]:
predicate = re_res.data
predicate

{'admitted_to': ['student', 'university'],
 'guess_correctness': ['student', 'student', 'university', 'BoolSort()']}

In [295]:
entities

{'person': ['Zhao Ming', 'Qian Hong', 'Sun Jie'],
 'university': ['Peking University',
  'Tsinghua University',
  'Beijing Normal University']}

In [296]:
import string

def generate_z3_declarations(entities):
    """
    将实体字典转换为 Z3 Python API 的声明代码字符串。
    
    参数:
    entities (dict): 键为类型名, 值为实体列表的字典。
    
    返回:
    str: 格式化后的 Z3 声明代码。
    """
    code_lines = []
    
    # 1. Z3 Type Declaration (类型声明)
    code_lines.append("# Z3 Type Declaration")
    for entity_type in entities.keys():
        line = f"{entity_type} = DeclareSort('{entity_type}')"
        code_lines.append(line)
    
    # 2. 常量定义
    code_lines.append("\n# 常量定义")
    for entity_type, names in entities.items():
        for name in names:
            # 处理空格，将实体名中的空格替换为下划线
            formatted_name = name.replace(" ", "_")
            line = f"{formatted_name} = Const('{formatted_name}', {entity_type})"
            code_lines.append(line)
            
    # 3. 变量定义 (a-z 顺序分配)
    code_lines.append("\n# 变量定义")
    # string.ascii_lowercase 提供 'abcdefghijklmnopqrstuvwxyz'
    alphabet = string.ascii_lowercase
    
    for i, entity_type in enumerate(entities.keys()):
        if i < len(alphabet):
            var_name = alphabet[i]
            line = f"{var_name} = Const('{var_name}', {entity_type})"
            code_lines.append(line)
        else:
            # 如果类型数量超过26个，可以在此处扩展逻辑（如 aa, ab...）
            break
            
    return "\n".join(code_lines)

# 测试数据
# entities = {
#     'students': ['Zhao Ming', 'Qian Hong', 'Sun Jie'],
#     'universities': ['Peking University', 'Tsinghua University', 'Beijing Normal University']
# }

# 调用函数并打印结果
z3_code = generate_z3_declarations(entities)
print(z3_code)

# Z3 Type Declaration
person = DeclareSort('person')
university = DeclareSort('university')

# 常量定义
Zhao_Ming = Const('Zhao_Ming', person)
Qian_Hong = Const('Qian_Hong', person)
Sun_Jie = Const('Sun_Jie', person)
Peking_University = Const('Peking_University', university)
Tsinghua_University = Const('Tsinghua_University', university)
Beijing_Normal_University = Const('Beijing_Normal_University', university)

# 变量定义
a = Const('a', person)
b = Const('b', university)


In [297]:
def generate_z3_functions(predicates):
    """
    将谓词字典转换为 Z3 Python API 的 Function 声明代码。
    
    参数:
    predicates (dict): 键为函数名，值为类型名称列表。
    
    返回:
    str: 格式化后的 Z3 Function 声明代码。
    """
    code_lines = []
    code_lines.append("# Z3 Function/Predicate Declaration")
    
    for func_name, types in predicates.items():
        # 将类型列表转换为逗号分隔的字符串
        # 注意：这里假设 types 列表中的元素如 'students' 是已经声明过的变量名
        # 对于 'BoolSort()' 这种带括号的，我们直接保持原样
        types_str = ", ".join(types)
        
        # 构造 Z3 Function 格式: name = Function('name', type1, type2, ...)
        line = f"{func_name} = Function('{func_name}', {types_str})"
        code_lines.append(line)
        
    return "\n".join(code_lines)

# # 测试数据
# predicates = {
#     'admitted_to': ['students', 'universities'],
#     'guess_correct_half': ['student', 'university', 'BoolSort()']
# }

# 调用函数并打印结果
z3_function_code = generate_z3_functions(predicate)
print(z3_function_code)

# Z3 Function/Predicate Declaration
admitted_to = Function('admitted_to', student, university)
guess_correctness = Function('guess_correctness', student, student, university, BoolSort())


# 3 Translation

In [330]:
translate_template = """

You are a Formal Logic Translator specializing in converting natural language reasoning into executable Z3 Python code. Your goal is to take a structured logical argument and map it onto a provided Z3 declaration schema.

Input Data:

    1.  Context & Question: The background information and the target problem.

    2.  Z3 Declarations: The predefined Sorts, Functions, and Predicates.

    3.  Reasoning Steps: A sequence of <step> blocks containing <premise> and <conclusion> in natural language.

Translation Rules (Strict Compliance Required):

    1.  Strict Symbolic Mapping:

        • Use ONLY the functions, predicates, and sorts provided in the Z3 Declaration.

        • Use ForAll, Exists, And, Or, Not, Implies, and == for logical connectors.

    2.  Explicit Expression Rule (No Pointers):

        • DO NOT refer to previous steps using variable names like conclusion_n or step_n.

        • Instead, you must explicitly restate the full Z3 logical expression of that conclusion as a premise in the new step.

    3.  Grounded Premises:

        • Every symbolic statement in premises_n must be derived from:

            ◦ The current step's natural language <premise>.

            ◦ A previous step's <conclusion> (written in full symbolic form).

            ◦ Global background knowledge or definitions implied by the Context.

    4.  Logical Purity:

        • Avoid using True or False as placeholders. Every statement should be a well-formed formula (WFF) using the provided predicates.

        • Ensure the conclusion_n is a direct logical consequence of the premises_n.

    5.  Output Format:

        • Output ONLY the Python code.

        • Structure the output as # Step N, followed by premises_n = [...] and conclusion_n = ....

        • Each step must be separated by exactly one newline.

Example of Expected Behavior:

Question:
<Context>Black Americans are twice as likely to suffer from hypertension as white Americans. The same is true when comparing Westernized black Africans to white Africans. The researchers hypothesized that the reason why westernized black people suffer from hypertension is the result of the interaction of two reasons: one is the high salt content of western foods, and the other is the adaptation mechanism of black genetic genes to the salt-deficient environment.</Context>
<Question>The following conclusions about contemporary westernized African blacks, if the item is true, can it best support the researchers' hypothesis?</Question>
<Options>
Option (A): The blood pressure of the descendants of Senegalese and Gambians is usually not high, and the history of Senegal and Gambia has not been short of salt.
Option (B): The unusually high salt intake in certain parts of Africa is a serious problem that threatens the health of residents.
Option (C): Considering health care, most African whites also pay attention to controlling salt intake.
Option (D): The blood pressure of Yoruba people in West Africa is not high. Yoruba people have lived inland far away from sea salt and far away from the Sahara salt mine in Africa.
</Options>

Z3 Declaration:
from z3 import *

# Declare sorts
Person = DeclareSort('Person')
Region = DeclareSort('Region')

# Declare predicates
Black = Function('Black', Person, BoolSort())
Westernized = Function('Westernized', Person, BoolSort())
FromRegion = Function('FromRegion', Person, Region, BoolSort())
SaltSufficientHistory = Function('SaltSufficientHistory', Region, BoolSort())
SaltDeficientHistory = Function('SaltDeficientHistory', Region, BoolSort())
GeneticAdaptationToLowSalt = Function('GeneticAdaptationToLowSalt', Person, BoolSort())
HighSaltDiet = Function('HighSaltDiet', Person, BoolSort())
Hypertension = Function('Hypertension', Person, BoolSort())

Reasoning steps:
<step>
<premise>Descendants of Senegalese (a) are Westernized black individuals.</premise>
<conclusion>Individual 'a' is Black and Westernized.</conclusion>
</step>
<step>
<premise>Individual 'a' is Black and Westernized.</premise>
<premise>The hypothesis states all Westernized Black people with high salt diets and genetic adaptation get hypertension.</premise>
<conclusion>Hypertension(a) == And(HighSaltDiet(a), GeneticAdaptation(a))</conclusion>
</step>

Example Output:
# Step 1
a = Const('a', Person)
premises_1 = [
    Black(a),
    Westernized(a)
]
conclusion_1 = And(Black(a), Westernized(a))

# Step 2=
premises_2 = [
    And(Black(a), Westernized(a)),
    ForAll([x], Implies(And(Black(x), Westernized(x)), Hypertension(x) == And(HighSaltDiet(x), GeneticAdaptation(x))))
]
conclusion_2 = (Hypertension(a) == And(HighSaltDiet(a), GeneticAdaptation(a)))
---

"""

## 3-1 Context Translation

In [328]:
rephrase_res

"<p1>Zhao Ming, Qian Hong, and Sun Jie were admitted to Peking University, Tsinghua University, and Beijing Normal University.</p1>\n<p2>Classmate A guessed that Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.</p2>\n<p3>Student B guessed that Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.</p3>\n<p4>Student C guessed that Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.</p4>\n<p5>The students' guesses were half correct.</p5>\n\n<atom_1> Each student's guess had exactly one correct statement.</atom_1>"

In [334]:
context_trans_template = """
You are an experienced logician and programmer, skilled at converting natural language into executable and correct Z3 API Python code.

## Task Description
Given the natural language problem background, problem description, options, and Z3 Declarations (Python code), your goal is to transform the problem background into correctly formatted and semantically accurate Python code.

## Input Data
- Context & Question: Background information and the target problem.
- Z3 Declarations: Predefined Sorts, Functions, and Predicates.

Translation Principle (Strict Compliance Required):
1. Strict Symbolic Mapping:
- Use ONLY the functions, predicates, and sorts provided in the Z3 Declaration.
- Use ForAll, Exists, And, Or, Not, Implies, iff, and == for logical connectors.
2. If new variables are needed, use the Const() function to create them, and specify their type. This type must be either predefined in the Declaration or a built-in Z3 type (BoolSort(), IntSort(), RealSort(), etc.).
3. Newly defined variables must not conflict with existing ones (do not use the same names).

## Output Format
1. IMPORTANT: The output must be executable Python code, strictly following Python syntax and Z3 API coding conventions;
2. The output is divided into two parts:
- New variable declarations: If new variables are needed, declare them using the Const function and specify their type, e.g., ```x = Const('x', student)```, where student is a type predefined in the Declaration code;
- Expression list: Translate each sentence into one or more expressions and place them in a list named context_fol.

---
Here is an example:

## Input
<Context>
<p1>Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens is the partners of the law firm.</p1>
<p2>Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens joined the firm between 1961 to 1968</p2>
<p3>Each partner joined the firm in different year.</p3>
<p4> Hodges joined earlier than Nader </p4>
<p5> King joined earlier than James. </p5>
<p6> Nader joined earlier than Gregg</p6>
<p7> James joined earlier than Gregg</p7>
<p8> Nader joined earlier than Owens</p8>
<p9> James joined earlier than MacNeil.</p9>
<p10>Gregg joined earlier than Ivan.</p10>

<axiom_1> If $a$ joined earlier than $b$, and $b$ joined earlier than $c$, then $a$ joined earlier than $c$.</axiom_1>
</Context>
<Question>
Which of the following statements cannot be true? </Question>
<Options>
(A) Hodges joined the firm in 1961.
(B) Hodges joined the firm in 1963.
(C) Gregg joined the firm in 1964.
(D) MacNeil joined the firm in 1964.
(E) Owens joined the firm in 1964.
</Options>

<Declaration>
# Z3 Type Declaration
partner = DeclareSort('partner')
year = DeclareSort('year')

# Constant Definition
Gregg = Const('Gregg', partner)
Hodges = Const('Hodges', partner)
Ivan = Const('Ivan', partner)
James = Const('James', partner)
King = Const('King', partner)
MacNeil = Const('MacNeil', partner)
Nader = Const('Nader', partner)
Owens = Const('Owens', partner)

# Variable Definition
a = Const('a', partner)
b = Const('b', year)

# Z3 Function/Predicate Declaration
join = Function('join', partner, year)

## Output
# Define needed variables
x, y, z = Consts('x y z', partner)
# Translated Python code
constrain_fol = [
    ForAll([x, y], Implies(x != y, JoinedYear(x) != JoinedYear(y))), # Each partner joined the firm in different year.
    join(Hodges) < join(Nader),  # Hodges joined earlier than Nader 
    join(King) < join(James),  # King joined earlier than James.
    join(Nader) < join(Gregg),  # Nader joined earlier than Gregg
    join(James) < join(Gregg), # James joined earlier than Gregg
    join(Nader) < join(Owens), # Nader joined earlier than Owens
    join(James) < join(MacNeil),  # James joined earlier than MacNeil.
    join(James) < join(MacNeil),  # Gregg joined earlier than Ivan.
    ForAll([x, y, z], Implies( And(JoinedEarlier(x, y), JoinedEarlier(y, z)), JoinedEarlier(x, z))), # axiom_1
]

----
Now, Please Translate the following content:
"""


## 3-2 Step Translation

In [327]:
premise_list, conclusion

(['Classmate A guessed Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.',
  'Classmate B guessed Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.',
  'Classmate C guessed Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.',
  "The students' guesses were half correct."],
 "At least one student's guess is partially correct.")

In [374]:
step_trans_template = """
You are an experienced logician and programmer, skilled at converting natural language into executable and correct Z3 API Python code.

## Task Description
Given the natural language problem context, problem description, a single reasoning step, and Z3 Declarations (Python code), your goal is to translate the given reasoning step into Z3-style Python code so that you can use Z3 to verify the logical validity of the step.

## Input Data
- Context & Question: Background information and the target problem.
- Z3 Declarations: Predefined Sorts, Functions, and Predicates.
- Reasoning Step: The given single reasoning step consists of two parts: (1) Premises: Each reasoning step contains multiple premises, each enclosed within ```<premise> </premise>``` blocks; (2) Conclusion: Similarly, the conclusion is enclosed within ```<conclusion> </conclusion>``` blocks. Your task is to translate both the premises and the conclusion into corresponding Z3 API-style Python code and place them in the lists `premise_fol` and `conclusion_fol`, respectively.

## Translation Principle (Strict Compliance Required):
1. Strict Symbolic Mapping:
- Use ONLY the functions, predicates, and sorts provided in the Z3 Declaration.
- Use ForAll, Exists, And, Or, Not, Implies, iff, and == for logical connectors.
2. If new variables are needed, use the Const() function to create them, then proceed with translation. When declaring new variables, specify their type, which must be either a type defined in the Declaration or a built-in Z3 type (BoolSort(), IntSort(), RealSort(), etc.).
3. Newly defined variables must not conflict with existing variables (do not use the same names).
4. IMPORTANT:
- Axioms MUST be translated: In addition to translating the premises, each reasoning step should also include the axioms from the context, which must be added to the `premise_fol` list.
- **Do NOT** translate the problem context or description into Z3 code! Translate **only the axioms and inference steps** into the corresponding Z3 code, and strictly adhere to the specified output format. This must be executable!
- Please Negate the conclusion for verifying its correctness！（VERY IMPORTANT）

## Output Format
1. IMPORTANT: The output must be executable Python code, strictly following Python syntax and Z3 API coding conventions.
2. The output is divided into two parts:
- Declaration of new variables: If new variables are needed, declare them using the Const function and specify their type, for example: ```x = Const('x', student)```, where `student` is a type already defined in the Declaration code.
- Expression lists: Each sentence must be translated into one or more expressions, and these expressions must be appended to the corresponding list.

---
Here is an example:

## Input
<Context>
<p1>Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens are the partners of the law firm.</p1>
<p2>Gregg, Hodges, Ivan, James, King, MacNeil, Nader, and Owens joined the firm between 1961 and 1968.</p2>
<p3>Each partner joined the firm in a different year.</p3>
<p4> Hodges joined earlier than Nader </p4>
<p5> King joined earlier than James. </p5>
<p6> Nader joined earlier than Gregg</p6>
<p7> James joined earlier than Gregg</p7>
<p8> Nader joined earlier than Owens</p8>
<p9> James joined earlier than MacNeil.</p9>
<p10>Gregg joined earlier than Ivan.</p10>

<Declaration>
```python
# Z3 Type Declaration
partner = DeclareSort('partner')
year = DeclareSort('year')

# Constant Definition
Gregg = Const('Gregg', partner)
Hodges = Const('Hodges', partner)
Ivan = Const('Ivan', partner)
James = Const('James', partner)
King = Const('King', partner)
MacNeil = Const('MacNeil', partner)
Nader = Const('Nader', partner)
Owens = Const('Owens', partner)

# Variable Definition
a = Const('a', partner)
b = Const('b', year)

# Z3 Function/Predicate Declaration
join = Function('join', partner, year)
```
</Declaration>
<ReasoningStep>
    <premise> Hodges joined earlier than Nader </premise>
    <premise> Nader joined earlier than Owens </premise>
    <conclusion>Hodges joined before Owens</conclusion>
</ReasoningStep>


## Output
```python
# Define required variables (ensure no conflict with Declaration)
x, y, z = Consts('x y z', partner)

# Translated premise Python code
premise_fol = [
    ForAll([x, y, z], Implies( And(JoinedEarlier(x, y), JoinedEarlier(y, z)), JoinedEarlier(x, z))), # axiom_1
    join(Hodges) < join(Nader),  # premise in reasoning step: Hodges joined earlier than Nader
    join(Nader) < join(Owens), # premise in reasoning step: Nader joined earlier than Owens
]

conclusion_fol = [
    Not(join(Hodges) < join(Owens)), # Negate the conclusion for verifying: Hodges doesn‘t joined earlier than Owens
]
```
Now, Please covert the following content to the python code we want:

<Context>
${context}
<Context>

<Declaration>
```python
${declaration}
```
</Declaration>

<ReasoningStep>
${step}
</ReasoningStep>

"""

from string import Template
transT = Template(step_trans_template)

In [361]:
rephrase_res

"<p1>Zhao Ming, Qian Hong, and Sun Jie were admitted to Peking University, Tsinghua University, and Beijing Normal University.</p1>\n<p2>Classmate A guessed that Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.</p2>\n<p3>Student B guessed that Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.</p3>\n<p4>Student C guessed that Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.</p4>\n<p5>The students' guesses were half correct.</p5>\n\n<atom_1> Each student's guess had exactly one correct statement.</atom_1>"

In [362]:
declaration_code = z3_code + z3_function_code
pprint( declaration_code )

('# Z3 Type Declaration\n'
 "person = DeclareSort('person')\n"
 "university = DeclareSort('university')\n"
 '\n'
 '# 常量定义\n'
 "Zhao_Ming = Const('Zhao_Ming', person)\n"
 "Qian_Hong = Const('Qian_Hong', person)\n"
 "Sun_Jie = Const('Sun_Jie', person)\n"
 "Peking_University = Const('Peking_University', university)\n"
 "Tsinghua_University = Const('Tsinghua_University', university)\n"
 "Beijing_Normal_University = Const('Beijing_Normal_University', university)\n"
 '\n'
 '# 变量定义\n'
 "a = Const('a', person)\n"
 "b = Const('b', university)# Z3 Function/Predicate Declaration\n"
 "admitted_to = Function('admitted_to', student, university)\n"
 "guess_correctness = Function('guess_correctness', student, student, "
 'university, BoolSort())')


In [356]:
step_list[0]

"<premise>Classmate A guessed Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.</premise>\n<premise>Classmate B guessed Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.</premise>\n<premise>Classmate C guessed Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.</premise>\n<premise>The students' guesses were half correct.</premise>\n<conclusion>At least one student's guess is partially correct.</conclusion>"

In [363]:
trans_input = transT.safe_substitute(context=rephrase_res, declaration=declaration_code, step=step_list[0])
trans_output = llm.generate(data=trans_input, args=args)
trans_output

"```python\n# Define required variables (ensure no conflict with Declaration)\nA, B, C = Consts('A B C', person)\nU1, U2, U3 = Consts('U1 U2 U3', university)\n\n# Translated premise Python code\npremise_fol = [\n    # A's guess\n    guess_correctness(A, Zhao_Ming, Tsinghua_University),\n    guess_correctness(A, Sun_Jie, Beijing_Normal_University),\n    \n    # B's guess\n    guess_correctness(B, Zhao_Ming, Beijing_Normal_University),\n    guess_correctness(B, Qian_Hong, Tsinghua_University),\n    \n    # C's guess\n    guess_correctness(C, Zhao_Ming, Peking_University),\n    guess_correctness(C, Sun_Jie, Tsinghua_University),\n    \n    # Half correctness condition\n    And(\n        Or(guess_correctness(A, Zhao_Ming, Tsinghua_University), guess_correctness(A, Sun_Jie, Beijing_Normal_University)),\n        Or(guess_correctness(B, Zhao_Ming, Beijing_Normal_University), guess_correctness(B, Qian_Hong, Tsinghua_University)),\n        Or(guess_correctness(C, Zhao_Ming, Peking_University), 

In [375]:
trans_input = transT.safe_substitute(context=rephrase_res, declaration=declaration_code, step=step_list[0])
trans_output = llm.generate(data=trans_input, args=args)
trans_output

"```python\n# Define required variables (ensure no conflict with Declaration)\nA, B, C = Consts('A B C', person)\nU1, U2, U3 = Consts('U1 U2 U3', university)\n\n# Translated premise Python code\npremise_fol = [\n    # A's guess: Zhao Ming was admitted to Tsinghua University and Sun Jie was admitted to Beijing Normal University.\n    guess_correctness(A, Zhao_Ming, Tsinghua_University),\n    guess_correctness(A, Sun_Jie, Beijing_Normal_University),\n\n    # B's guess: Zhao Ming was admitted to Beijing Normal University and Qian Hong was admitted to Tsinghua University.\n    guess_correctness(B, Zhao_Ming, Beijing_Normal_University),\n    guess_correctness(B, Qian_Hong, Tsinghua_University),\n\n    # C's guess: Zhao Ming was admitted to Peking University and Sun Jie was admitted to Tsinghua University.\n    guess_correctness(C, Zhao_Ming, Peking_University),\n    guess_correctness(C, Sun_Jie, Tsinghua_University),\n\n    # The students' guesses were half correct.\n    And(\n        Or(gu

In [376]:
pprint(trans_output)

('```python\n'
 '# Define required variables (ensure no conflict with Declaration)\n'
 "A, B, C = Consts('A B C', person)\n"
 "U1, U2, U3 = Consts('U1 U2 U3', university)\n"
 '\n'
 '# Translated premise Python code\n'
 'premise_fol = [\n'
 "    # A's guess: Zhao Ming was admitted to Tsinghua University and Sun Jie "
 'was admitted to Beijing Normal University.\n'
 '    guess_correctness(A, Zhao_Ming, Tsinghua_University),\n'
 '    guess_correctness(A, Sun_Jie, Beijing_Normal_University),\n'
 '\n'
 "    # B's guess: Zhao Ming was admitted to Beijing Normal University and "
 'Qian Hong was admitted to Tsinghua University.\n'
 '    guess_correctness(B, Zhao_Ming, Beijing_Normal_University),\n'
 '    guess_correctness(B, Qian_Hong, Tsinghua_University),\n'
 '\n'
 "    # C's guess: Zhao Ming was admitted to Peking University and Sun Jie was "
 'admitted to Tsinghua University.\n'
 '    guess_correctness(C, Zhao_Ming, Peking_University),\n'
 '    guess_correctness(C, Sun_Jie, Tsinghua_Univer

In [379]:
trans_code = extract_python_block(trans_output)
pprint(trans_code)

('# Define required variables (ensure no conflict with Declaration)\n'
 "A, B, C = Consts('A B C', person)\n"
 "U1, U2, U3 = Consts('U1 U2 U3', university)\n"
 '\n'
 '# Translated premise Python code\n'
 'premise_fol = [\n'
 "    # A's guess: Zhao Ming was admitted to Tsinghua University and Sun Jie "
 'was admitted to Beijing Normal University.\n'
 '    guess_correctness(A, Zhao_Ming, Tsinghua_University),\n'
 '    guess_correctness(A, Sun_Jie, Beijing_Normal_University),\n'
 '\n'
 "    # B's guess: Zhao Ming was admitted to Beijing Normal University and "
 'Qian Hong was admitted to Tsinghua University.\n'
 '    guess_correctness(B, Zhao_Ming, Beijing_Normal_University),\n'
 '    guess_correctness(B, Qian_Hong, Tsinghua_University),\n'
 '\n'
 "    # C's guess: Zhao Ming was admitted to Peking University and Sun Jie was "
 'admitted to Tsinghua University.\n'
 '    guess_correctness(C, Zhao_Ming, Peking_University),\n'
 '    guess_correctness(C, Sun_Jie, Tsinghua_University),\n'
 '\n'

In [382]:
def wrap_z3_code(declaration, expression):
    z3_code = ""
    z3_code += "from z3 import *\n\n"
    z3_code += "s = Solver()\n\n"
    z3_code += "s.reset()"
    z3_code += "# --- Declarations ---\n\n"
    z3_code += declaration + "\n\n"
    z3_code += "# --- Expressions ---\n\n"
    z3_code += expression + "\n\n"
    z3_code += "s.add(premise_fol)\n\n"
    z3_code += "s.add(conclusion_fol)\n\n"
    z3_code += "result = s.check()\n"
    z3_code += "print(f'Result: {result}')\n"
    z3_code += "if result == sat:\n"
    z3_code += "    print('Model:', s.model())\n"
    return z3_code

In [383]:
z3_code_exe = wrap_z3_code(declaration_code, trans_code)

In [385]:
pprint(z3_code_exe)

('from z3 import *\n'
 '\n'
 's = Solver()\n'
 '\n'
 's.reset()# --- Declarations ---\n'
 '\n'
 '# Z3 Type Declaration\n'
 "person = DeclareSort('person')\n"
 "university = DeclareSort('university')\n"
 '\n'
 '# 常量定义\n'
 "Zhao_Ming = Const('Zhao_Ming', person)\n"
 "Qian_Hong = Const('Qian_Hong', person)\n"
 "Sun_Jie = Const('Sun_Jie', person)\n"
 "Peking_University = Const('Peking_University', university)\n"
 "Tsinghua_University = Const('Tsinghua_University', university)\n"
 "Beijing_Normal_University = Const('Beijing_Normal_University', university)\n"
 '\n'
 '# 变量定义\n'
 "a = Const('a', person)\n"
 "b = Const('b', university)# Z3 Function/Predicate Declaration\n"
 "admitted_to = Function('admitted_to', student, university)\n"
 "guess_correctness = Function('guess_correctness', student, student, "
 'university, BoolSort())\n'
 '\n'
 '# --- Expressions ---\n'
 '\n'
 '# Define required variables (ensure no conflict with Declaration)\n'
 "A, B, C = Consts('A B C', person)\n"
 "U1, U2, U3 

In [389]:
res = run_code(z3_code_exe)

In [390]:
res

{'success': False,
 'output': '',
 'error': 'Traceback (most recent call last):\n  File "<string>", line 22, in <module>\nNameError: name \'student\' is not defined\n'}

In [393]:
correct_loop(z3_code_exe)

{'success': False, 'output': '', 'error': 'Traceback (most recent call last):\n  File "<string>", line 22, in <module>\nNameError: name \'student\' is not defined\n'}
The error in the provided Python code is due to the undefined variable `student`. This variable should be declared before it is used in the function declarations and expressions. Additionally, the variable names `A`, `B`, `C`, `U1`, `U2`, and `U3` should be consistent with the previously declared constants.

Here is the corrected Python code:

```python
from z3 import *

s = Solver()

s.reset()  # --- Declarations ---

# Z3 Type Declaration
person = DeclareSort('person')
university = DeclareSort('university')

# Constants definition
Zhao_Ming = Const('Zhao_Ming', person)
Qian_Hong = Const('Qian_Hong', person)
Sun_Jie = Const('Sun_Jie', person)
Peking_University = Const('Peking_University', university)
Tsinghua_University = Const('Tsinghua_University', university)
Beijing_Normal_University = Const('Beijing_Normal_Universit